# Measure reconstruction error before claiming speed

Prerequisites: Rounding, dynamic range and numerical error.

Symmetric quantization chooses scale=max|w|/(2^(b−1)−1), rounds w/scale to an integer and reconstructs by multiplying by scale. With no clipping beyond the chosen range, nearest rounding limits absolute reconstruction error to approximately half a step.

The notebook quantizes four fixed weights at 8 and 4 bits. This checks numerical reconstruction, not model accuracy or speed. An implementation must actually use suitable integer kernels, account for scales and packing, and evaluate downstream behavior before claiming memory or latency improvements. Outliers can expand the shared scale and worsen resolution for small weights.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Scale | max|w| / integer limit | Connect ranges |
| Round | Integer code | Lose precision |
| Reconstruct | code × scale | Measure numeric error |

## Predict

At 8 bits the maximum integer magnitude is 127. The observed reconstruction error must be no larger than scale/2, apart from numerical tolerance.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import extensions as m

In [ ]:
result=m.quantization()
assert result['maximum_error']<=result['half_step']+1e-12
print(result)
print('Four bits:',m.quantization(4))

## Perturb and explain

Add one large outlier and compare errors on the smaller weights. Explain when per-channel scaling might help and what metadata it adds.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

A small weight-reconstruction error proves lower inference latency:

1. Yes
2. No; kernels and workload must be measured
3. Only because the values are integers

Answer and explanations: 1: Numerical error is not a timing measurement. 2: Hardware and implementation determine speed. 3: Integer labels alone do not select an optimized kernel.

## Primary evidence

- [Hugging Face — Quantization concepts](https://huggingface.co/docs/optimum/en/concept_guides/quantization)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.